In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pd.read_csv("database/train_mod_tratado.csv")

X = df.drop(columns=["Preco"])
y = df["Preco"]

In [3]:
cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [4]:
preprocessor_ridge = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols)
])

pipe_ridge = Pipeline([
    ("prep", preprocessor_ridge),
    ("model", Ridge())
])

param_grid_ridge = {
    "model__alpha": [0.001, 0.01, 0.1, 1, 10, 100]
}

In [5]:
preprocessor_tree = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols)
])

pipe_rf = Pipeline([
    ("prep", preprocessor_tree),
    ("model", RandomForestRegressor(random_state=42))
])

param_grid_rf = {
    "model__n_estimators": [50, 100, 150, 200, 250, 300],
    "model__max_depth": [None, 10, 20, 30, 40],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

In [6]:
pipe_gb = Pipeline([
    ("prep", preprocessor_tree),
    ("model", GradientBoostingRegressor(random_state=42))
])

param_grid_gb = {
    "model__n_estimators": [50, 100, 150, 200, 250, 300],
    "model__learning_rate": [0.001, 0.005, 0.01, 0.05, 0.1],
    "model__max_depth": [2, 3, 5, 10],
    "model__subsample": [0.8, 1.0]
}

In [7]:
grid_ridge = GridSearchCV(
    pipe_ridge,
    param_grid_ridge,
    cv=10,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_rf = GridSearchCV(
    pipe_rf,
    param_grid_rf,
    cv=10,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_gb = GridSearchCV(
    pipe_gb,
    param_grid_gb,
    cv=10,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)   

grid_ridge.fit(X_train, y_train)
grid_rf.fit(X_train, y_train)
grid_gb.fit(X_train, y_train)

GridSearchCV(cv=10,
             estimator=Pipeline(steps=[('prep',
                                        ColumnTransformer(transformers=[('cat',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         ['Fabricante',
                                                                          'Modelo',
                                                                          'Categoria',
                                                                          'Couro',
                                                                          'Combustivel',
                                                                          'Tipo_cambio',
                                                                          'Tração',
                                                                          'Portas',
                                                                          'Cor',
                                                                          'Adesivos_personalizados',
                                                                          'Radio_AM_FM']),
                                                                        ('num',
                                                                         'passthrough',
                                                                         ['Débitos',
                                                                          'Ano',
                                                                          'Volume_motor',
                                                                          'Km',
                                                                          'Cilindros',
                                                                          'Rodas',
                                                                          'Airbags',
                                                                          'Numero_proprietarios',
                                                                          'Historico_troca_oleo'])])),
                                       ('model',
                                        GradientBoostingRegressor(random_state=42))]),
             n_jobs=-1,
             param_grid={'model__learning_rate': [0.001, 0.005, 0.01, 0.05,
                                                  0.1],
                         'model__max_depth': [2, 3, 5, 10],
                         'model__n_estimators': [50, 100, 150, 200, 250, 300],
                         'model__subsample': [0.8, 1.0]},
             scoring='neg_root_mean_squared_error')

In [8]:
import os
import joblib

os.makedirs("models", exist_ok=True)

joblib.dump(grid_ridge.best_estimator_, "models/ridge_model.pkl")
joblib.dump(grid_rf.best_estimator_, "models/random_forest_model.pkl")
joblib.dump(grid_gb.best_estimator_, "models/gradient_boosting_model.pkl")

print("Modelos salvos com sucesso.")

Modelos salvos com sucesso.


In [9]:
resultados = {}

for nome, grid in {
    "Ridge": grid_ridge,
    "Random Forest": grid_rf,
    "Gradient Boosting": grid_gb
}.items():
    
    y_pred = grid.best_estimator_.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)
    
    resultados[nome] = {
        "Melhores hiperparâmetros": grid.best_params_,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

resultados

{'Ridge': {'Melhores hiperparâmetros': {'model__alpha': 10},
  'MAE': 10051.24035832366,
  'RMSE': 16050.688430516948,
  'R2': 0.291237974032192},
 'Random Forest': {'Melhores hiperparâmetros': {'model__max_depth': 30,
   'model__min_samples_leaf': 2,
   'model__min_samples_split': 5,
   'model__n_estimators': 100},
  'MAE': 5849.034341913498,
  'RMSE': 11600.658352041595,
  'R2': 0.6297642215714869},
 'Gradient Boosting': {'Melhores hiperparâmetros': {'model__learning_rate': 0.05,
   'model__max_depth': 5,
   'model__n_estimators': 300,
   'model__subsample': 0.8},
  'MAE': 6980.110092151477,
  'RMSE': 12612.46916753349,
  'R2': 0.5623636904886441}}

In [10]:
resultado_df = pd.DataFrame(resultados).T
resultado_df

,Melhores hiperparâmetros,MAE,RMSE,R2
Ridge,{'model__alpha': 10},10051.240358,16050.688431,0.291238
Random Forest,"{'model__max_depth': 30, 'model__min_samples_l...",5849.034342,11600.658352,0.629764
Gradient Boosting,"{'model__learning_rate': 0.05, 'model__max_dep...",6980.110092,12612.469168,0.562364
